In [1]:
!git clone https://github.com/nttng207/TwinLiteNetPlus.git --branch develop/tantran

Cloning into 'TwinLiteNetPlus'...
remote: Enumerating objects: 180, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 180 (delta 65), reused 59 (delta 46), pack-reused 93 (from 1)
Receiving objects: 100% (180/180), 11.72 MiB | 33.53 MiB/s, done.
Resolving deltas: 100% (80/80), done.


In [2]:
!pip install -q gdown
!mkdir -p /kaggle/working/TwinLiteNetPlus/pretrained

!gdown --folder "https://drive.google.com/drive/folders/1EqBzUw0b17aEumZmWYrGZmbx_XJqU-vz" \
-O /kaggle/working/TwinLiteNetPlus/pretrained

Retrieving folder contents
Processing file 1H8P-GrOUBOaVs5LEqXBfz0dC9gguUUio large.pth
Processing file 121z9XUh7_lgze8i6nS6Ne2HQ7ZG5_uG9 medium.pth
Processing file 1SvD03WZOq8eN4X0bMFNZyzj4upZM1QVl nano.pth
Processing file 1N7jNsa8P1dM-UevqPRsx6heUpN4L00FJ small.pth
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1H8P-GrOUBOaVs5LEqXBfz0dC9gguUUio
To: /kaggle/working/TwinLiteNetPlus/pretrained/large.pth
100%|██████████████████████████████████████| 7.96M/7.96M [00:00<00:00, 30.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=121z9XUh7_lgze8i6nS6Ne2HQ7ZG5_uG9
To: /kaggle/working/TwinLiteNetPlus/pretrained/medium.pth
100%|███████████████████████████████████████| 2.05M/2.05M [00:00<00:00, 192MB/s]
Downloading...
From: https://drive.google.com/uc?id=1SvD03WZOq8eN4X0bMFNZyzj4upZM1QVl
To: /kaggle/working/TwinLiteNetPlus/pretrained/nano.pth
100%|████████████████████████████████

In [3]:
!mkdir -p /kaggle/working/TwinLiteNetPlus/finetune

!gdown --folder "https://drive.google.com/drive/folders/1AZ7egoq7iVOYgkqjSE6hdkrFWsj1Pbwt" \
-O /kaggle/working/TwinLiteNetPlus/finetune

Retrieving folder contents
Processing file 1-WRSYRdRTUM3rgB72Mo_c4M2m_azUzqt large_cityscapes_weights.pth
Processing file 1YKY6mSsM7sm2pNGBqhoCxpG0rjUAQmAv medium_cityscapes_weights.pth
Processing file 17dEum1A7Sa7v7gWXzJzyA3de1qFpcRom nano_cityscapes_weights.pth
Processing file 1s5nZhCOevdyRD7ZjeMo5wkPjQEm1JsWz small_cityscapes_weights.pth
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1-WRSYRdRTUM3rgB72Mo_c4M2m_azUzqt
To: /kaggle/working/TwinLiteNetPlus/finetune/large_cityscapes_weights.pth
100%|██████████████████████████████████████| 7.95M/7.95M [00:00<00:00, 27.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1YKY6mSsM7sm2pNGBqhoCxpG0rjUAQmAv
To: /kaggle/working/TwinLiteNetPlus/finetune/medium_cityscapes_weights.pth
100%|███████████████████████████████████████| 2.05M/2.05M [00:00<00:00, 217MB/s]
Downloading...
From: https://drive.google.com/uc?id=17dEum1A7Sa7v7gWX

## Copy Cityscapes dataset loader and val script into the repo

In [4]:
import shutil
# Copy the two Cityscapes files from Kaggle input (attached as dataset) into the repo.
# If you uploaded them as a dataset attachment, adjust the source path below.
# Otherwise the cell below writes them inline.
repo = "/kaggle/working/TwinLiteNetPlus"
print("Files will be written in the next cell.")

Files will be written in the next cell.


In [5]:
cityscapes_code = r'''
import os
import glob
import numpy as np
import cv2
from PIL import Image
import torch
import torch.utils.data
import torchvision.transforms as transforms


# Cityscapes gtFine_labelIds.png raw label IDs (0-33)
DRIVABLE_IDS = [7]        # road
LANE_IDS     = [6, 0]     # ground / road markings (label 0 = unlabeled is NOT lane,
                           # but we keep road marking = any non-road flat marking)
# More precise: Cityscapes lane-relevant IDs
# 6  = ground (includes road markings in some versions)
# We will use only markings visible in gtFine_labelIds:
# Actually the standard approach: road=7 for DA, lane lines don't have a dedicated
# single ID in labelIds — they're part of road. We use label 6 (ground) as proxy.
# This matches the spirit of the BDD100K approach used in TwinLiteNet.
DRIVABLE_IDS = [7]
LANE_IDS     = [6]


def letterbox(im, new_shape=(384, 640), color=(114, 114, 114)):
    """Resize + pad to new_shape (H, W), same as MAPILLARY.py."""
    shape = im.shape[:2]  # (H, W)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    new_unpad = (int(round(shape[1] * r)), int(round(shape[0] * r)))  # (W, H)
    dw = new_shape[1] - new_unpad[0]
    dh = new_shape[0] - new_unpad[1]
    dw /= 2
    dh /= 2
    if shape[::-1] != new_unpad:
        im = cv2.resize(im, new_unpad, interpolation=cv2.INTER_LINEAR)
    top    = int(round(dh - 0.1))
    bottom = int(round(dh + 0.1))
    left   = int(round(dw - 0.1))
    right  = int(round(dw + 0.1))
    im = cv2.copyMakeBorder(im, top, bottom, left, right,
                             cv2.BORDER_CONSTANT, value=color)
    return im


class CityscapesDataset(torch.utils.data.Dataset):
    """
    Cityscapes dataset for TwinLiteNetPlus — mirrors MapillaryDataset interface exactly.

    Returns: (img_path, image_tensor, (seg_da, seg_ll))
        img_path   : str
        image      : uint8 Tensor (3, 384, 640), BGR, values 0-255
        seg_da     : float Tensor (2, 360, 640)  one-hot drivable area
        seg_ll     : float Tensor (2, 360, 640)  one-hot lane line

    Folder layout under data_root:
        leftImg8bit/val/<city>/*_leftImg8bit.png
        gtFine_trainvaltest/gtFine/val/<city>/*_gtFine_labelIds.png
    """

    W_, H_      = 640, 384   # final image size (matches letterbox)
    MASK_W      = 640        # label width  (matches MAPILLARY.py)
    MASK_H      = 360        # label height (matches MAPILLARY.py — NOT 384)

    def __init__(self, hyp: dict, root: str, split: str = "val", valid: bool = True):
        super().__init__()
        self.root  = root
        self.split = split
        self.valid = valid
        self.Tensor = transforms.ToTensor()

        img_dir = os.path.join(root, "leftImg8bit", split)
        gt_dir  = os.path.join(root, "gtFine_trainvaltest", "gtFine", split)

        self.img_paths = sorted(
            glob.glob(os.path.join(img_dir, "*", "*_leftImg8bit.png"))
        )
        self.gt_paths = []
        for img_path in self.img_paths:
            city      = os.path.basename(os.path.dirname(img_path))
            base      = os.path.basename(img_path).replace("_leftImg8bit.png", "")
            mask_path = os.path.join(gt_dir, city, f"{base}_gtFine_labelIds.png")
            self.gt_paths.append(mask_path)

        missing = [p for p in self.gt_paths if not os.path.exists(p)]
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} label files not found. First: {missing[0]}\n"
                "Check --data_root contains 'leftImg8bit/' and 'gtFine_trainvaltest/'."
            )
        if not self.img_paths:
            raise RuntimeError(f"No images found under {img_dir}")

        print(f"[CityscapesDataset] {len(self.img_paths)} images | split='{split}' | "
              f"image=({self.H_},{self.W_}) mask=({self.MASK_H},{self.MASK_W})")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]

        # ---------- image ----------
        # Read as BGR uint8 (same as cv2.imread in MAPILLARY.py)
        image = cv2.imread(img_path)                          # (H, W, 3) BGR uint8
        image = letterbox(image, (self.H_, self.W_))          # (384, 640, 3)

        # ---------- label ----------
        label = np.array(Image.open(self.gt_paths[idx]), dtype=np.int32)  # (H, W)

        # Binary masks: 255 = positive, 0 = negative  (matches MAPILLARY.py)
        label1 = np.isin(label, DRIVABLE_IDS).astype(np.uint8) * 255   # drivable
        label2 = np.isin(label, LANE_IDS    ).astype(np.uint8) * 255   # lane

        label1 = cv2.resize(label1, (self.MASK_W, self.MASK_H), interpolation=cv2.INTER_NEAREST)
        label2 = cv2.resize(label2, (self.MASK_W, self.MASK_H), interpolation=cv2.INTER_NEAREST)

        # One-hot encode: (2, MASK_H, MASK_W) — same cv2.threshold logic as MAPILLARY.py
        _, seg_b1 = cv2.threshold(label1, 1, 255, cv2.THRESH_BINARY_INV)
        _, seg_b2 = cv2.threshold(label2, 1, 255, cv2.THRESH_BINARY_INV)
        _, seg1   = cv2.threshold(label1, 1, 255, cv2.THRESH_BINARY)
        _, seg2   = cv2.threshold(label2, 1, 255, cv2.THRESH_BINARY)

        seg1   = self.Tensor(seg1)    # (1, MASK_H, MASK_W)
        seg2   = self.Tensor(seg2)
        seg_b1 = self.Tensor(seg_b1)
        seg_b2 = self.Tensor(seg_b2)

        seg_da = torch.stack((seg_b1[0], seg1[0]), 0)   # (2, MASK_H, MASK_W)
        seg_ll = torch.stack((seg_b2[0], seg2[0]), 0)

        # image: HWC BGR → CHW, keep uint8 (val() divides by 255 itself)
        image = image[:, :, ::-1].transpose(2, 0, 1)    # RGB, (3, 384, 640)
        image = np.ascontiguousarray(image)

        return img_path, torch.from_numpy(image), (seg_da, seg_ll)

'''
with open("/kaggle/working/TwinLiteNetPlus/CITYSCAPES.py", "w") as f:
    f.write(cityscapes_code)
print("CITYSCAPES.py written.")

CITYSCAPES.py written.


In [6]:
val_code = r'''
import torch
import torch.backends.cudnn as cudnn
import torch.optim.lr_scheduler
import yaml
from argparse import ArgumentParser
from model.model import TwinLiteNetPlus
from utils import val, netParams
from CITYSCAPES import CityscapesDataset


def validation(args):
    """
    Perform model validation on the Cityscapes dataset.
    :param args: Parsed command-line arguments.
    """

    # Initialize model
    model = TwinLiteNetPlus(args)
    cuda_available = torch.cuda.is_available()
    if cuda_available:
        model = model.cuda()
        cudnn.benchmark = True

    # Load hyperparameters from YAML file
    with open(args.hyp, errors='ignore') as f:
        hyp = yaml.safe_load(f)

    # Create validation data loader
    val_set = CityscapesDataset(
        hyp=hyp,
        root=args.data_root,
        split="val",
        valid=True
    )
    valLoader = torch.utils.data.DataLoader(
        val_set,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=True
    )

    # Print model parameter count
    print(f'Total network parameters: {netParams(model)}')

    # Load pretrained weights
    model.load_state_dict(torch.load(args.weight))
    model.eval()

    # Perform validation
    da_segment_results, ll_segment_results = val(valLoader, model, args.half, args=args)

    # Print results
    print(f"Driving Area Segment: mIOU({da_segment_results[2]:.3f})")
    print(f"Lane Line Segment: Acc({ll_segment_results[0]:.3f}) IOU({ll_segment_results[1]:.3f})")


if __name__ == '__main__':
    parser = ArgumentParser()
    parser.add_argument('--weight', type=str, default="pretrained/large.pth",
                        help='Path to model weights')
    parser.add_argument('--num_workers', type=int, default=12,
                        help='Number of parallel threads')
    parser.add_argument('--batch_size', type=int, default=16,
                        help='Batch size for validation')
    parser.add_argument('--config', type=str,
                        choices=["nano", "small", "medium", "large"],
                        help='Model configuration')
    parser.add_argument('--hyp', type=str,
                        default='./hyperparameters/twinlitev2_hyper.yaml',
                        help='Path to hyperparameters YAML file')
    parser.add_argument('--half', action='store_true',
                        help='Use half precision for inference')
    parser.add_argument('--verbose', action='store_true',
                        help='Enable verbose logging')
    parser.add_argument('--data_root', type=str,
                        help='Path to the Cityscapes dataset root '
                             '(must contain leftImg8bit/ and gtFine_trainvaltest/)')

    validation(parser.parse_args())

'''
with open("/kaggle/working/TwinLiteNetPlus/val_CITYSCAPES.py", "w") as f:
    f.write(val_code)
print("val_CITYSCAPES.py written.")

val_CITYSCAPES.py written.


In [7]:
train_code = r'''
import os
import time
import torch
import torch.optim.lr_scheduler
import torch.backends.cudnn as cudnn
import yaml
import math
from copy import deepcopy
from argparse import ArgumentParser

from model.model import TwinLiteNetPlus
from loss import TotalLoss
from utils import train, val, netParams, save_checkpoint, poly_lr_scheduler
from CITYSCAPES import CityscapesDataset  # <-- changed


class ModelEMA:
    """Exponential Moving Average (EMA) for model parameters"""
    def __init__(self, model, decay=0.9999, updates=0):
        self.ema = deepcopy(model).eval()
        self.updates = updates
        self.decay = lambda x: decay * (1 - math.exp(-x / 2000))
        for p in self.ema.parameters():
            p.requires_grad_(False)

    def update(self, model):
        with torch.no_grad():
            self.updates += 1
            d = self.decay(self.updates)
            msd = model.state_dict()
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point:
                    v *= d
                    v += (1. - d) * msd[k].detach()


def train_net(args, hyp):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    cudnn.benchmark = True

    model = TwinLiteNetPlus(args).to(device)
    print("Total params:", netParams(model))

    train_set = CityscapesDataset(   # <-- changed
        hyp=hyp,
        root=args.data_root,
        split="train",               # <-- changed (Cityscapes uses "train" not "training")
        valid=False
    )

    val_set = CityscapesDataset(     # <-- changed
        hyp=hyp,
        root=args.data_root,
        split="val",                 # <-- changed (Cityscapes uses "val" not "validation")
        valid=True
    )

    train_loader = torch.utils.data.DataLoader(
        train_set,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=True,
        drop_last=True,
        persistent_workers=True,
        prefetch_factor=4
    )

    val_loader = torch.utils.data.DataLoader(
        val_set,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=True,
    )

    criterion = TotalLoss(hyp)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=hyp["lr"],
        betas=(hyp["momentum"], 0.999),
        eps=hyp["eps"],
        weight_decay=hyp["weight_decay"]
    )

    scaler = torch.amp.GradScaler("cuda")
    ema = ModelEMA(model) if args.ema else None

    start_epoch = 0
    if args.resume and os.path.isfile(args.resume):
        ckpt = torch.load(args.resume, map_location="cpu")
        if isinstance(ckpt, dict) and "state_dict" in ckpt:
            print("=> Resuming full checkpoint")
            model.load_state_dict(ckpt["state_dict"], strict=True)
            optimizer.load_state_dict(ckpt["optimizer"])
            start_epoch = ckpt.get("epoch", 0)
            if args.ema and ckpt.get("ema_state_dict") is not None:
                ema.ema.load_state_dict(ckpt["ema_state_dict"])
                ema.updates = ckpt.get("updates", 0)
        else:
            print("=> Loading pretrained weights only (finetune)")
            model.load_state_dict(ckpt, strict=False)
            start_epoch = 0

    os.makedirs(args.savedir, exist_ok=True)

    for epoch in range(start_epoch, args.max_epochs):
        poly_lr_scheduler(args, hyp, optimizer, epoch)

        model.train()
        start_train = time.time()
        train(args, train_loader, model, criterion, optimizer, epoch, scaler, False, ema)
        print(f"Epoch {epoch} training time: {time.time() - start_train:.2f}s")

        model.eval()
        start_val = time.time()
        da_res, ll_res = val(val_loader, ema.ema if args.ema else model, args=args)
        print(f"Epoch {epoch} validation time: {time.time() - start_val:.2f}s")
        print(
            f"[{epoch}] "
            f"DA mIoU: {da_res[2]:.4f} | "
            f"LL Acc: {ll_res[0]:.4f} IOU: {ll_res[1]:.4f}"
        )

        save_checkpoint({
            "epoch": epoch + 1,
            "state_dict": model.state_dict(),
            "ema_state_dict": ema.ema.state_dict() if args.ema else None,
            "updates": ema.updates if args.ema else None,
            "optimizer": optimizer.state_dict(),
        }, os.path.join(args.savedir, "checkpoint.pth.tar"))


if __name__ == '__main__':
    parser = ArgumentParser()
    parser.add_argument('--max_epochs',  type=int, default=100)
    parser.add_argument('--num_workers', type=int, default=12)
    parser.add_argument('--batch_size',  type=int, default=16)
    parser.add_argument('--savedir',     default='./testv3')
    parser.add_argument('--hyp',         type=str, default='./hyperparameters/twinlitev2_hyper.yaml')
    parser.add_argument('--resume',      type=str, default='')
    parser.add_argument('--config',      default='nano')
    parser.add_argument('--verbose',     action='store_true')
    parser.add_argument('--ema',         action='store_true')
    parser.add_argument('--data_root',   type=str,
                        help='Path to Cityscapes root (contains leftImg8bit/ and gtFine_trainvaltest/)')

    args = parser.parse_args()

    with open(args.hyp, errors='ignore') as f:
        hyp = yaml.safe_load(f)

    train_net(args, hyp.copy())

'''
with open("/kaggle/working/TwinLiteNetPlus/train_CITYSCAPES.py", "w") as f:
    f.write(train_code)
print("train_CITYSCAPES.py written.")

train_CITYSCAPES.py written.


In [8]:
%cd /kaggle/working/TwinLiteNetPlus

/kaggle/working/TwinLiteNetPlus


# Model Version

## Nano

### Validate with Base

In [9]:
!python val_CITYSCAPES.py --config 'nano' --weight '/kaggle/working/TwinLiteNetPlus/pretrained/nano.pth' \
                --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
                --num_workers 4 \
                --batch_size 16

[CityscapesDataset] 500 images | split='val' | image=(384,640) mask=(360,640)
Total network parameters: 33379
Driving Area Segment: mIOU(0.811)
Lane Line Segment: Acc(0.514) IOU(0.009)


### Finetuning

In [10]:
# !python train_CITYSCAPES.py \
#     --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
#     --hyp "./hyperparameters/twinlitev2_hyper.yaml" \
#     --savedir "./finetune_cityscapes/nano" \
#     --resume "/kaggle/working/TwinLiteNetPlus/pretrained/nano.pth" \
#     --config "nano" \
#     --max_epochs 40 \
#     --batch_size 32 \
#     --num_workers 4 \
#     --ema \
#     --verbose

### Validate with Finetuned

In [11]:
!python val_CITYSCAPES.py --config 'nano' --weight '/kaggle/working/TwinLiteNetPlus/finetune/nano_cityscapes_weights.pth' \
                --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
                --num_workers 4 \
                --batch_size 16

[CityscapesDataset] 500 images | split='val' | image=(384,640) mask=(360,640)
Total network parameters: 33379
/kaggle/working/TwinLiteNetPlus/IOUEval.py:112: RuntimeWarning: invalid value encountered in divide
  IoU = intersection / union
/kaggle/working/TwinLiteNetPlus/IOUEval.py:104: RuntimeWarning: invalid value encountered in divide
  IoU = intersection / union
Driving Area Segment: mIOU(0.886)
Lane Line Segment: Acc(0.512) IOU(0.019)


## Small

### Validate with Base

In [12]:
!python val_CITYSCAPES.py --config 'small' --weight '/kaggle/working/TwinLiteNetPlus/pretrained/small.pth' \
                --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
                --num_workers 4 \
                --batch_size 16

[CityscapesDataset] 500 images | split='val' | image=(384,640) mask=(360,640)
Total network parameters: 121552
Driving Area Segment: mIOU(0.797)
Lane Line Segment: Acc(0.515) IOU(0.010)


### Finetuning

In [13]:
# !python train_CITYSCAPES.py \
#     --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
#     --hyp "./hyperparameters/twinlitev2_hyper.yaml" \
#     --savedir "./finetune_cityscapes/small" \
#     --resume "/kaggle/working/TwinLiteNetPlus/pretrained/small.pth" \
#     --config "small" \
#     --max_epochs 40 \
#     --batch_size 32 \
#     --num_workers 4 \
#     --ema \
#     --verbose

### Validate with Finetuned

In [14]:
!python val_CITYSCAPES.py --config 'small' --weight '/kaggle/working/TwinLiteNetPlus/finetune/small_cityscapes_weights.pth' \
                --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
                --num_workers 4 \
                --batch_size 16

[CityscapesDataset] 500 images | split='val' | image=(384,640) mask=(360,640)
Total network parameters: 121552
/kaggle/working/TwinLiteNetPlus/IOUEval.py:112: RuntimeWarning: invalid value encountered in divide
  IoU = intersection / union
/kaggle/working/TwinLiteNetPlus/IOUEval.py:104: RuntimeWarning: invalid value encountered in divide
  IoU = intersection / union
Driving Area Segment: mIOU(0.910)
Lane Line Segment: Acc(0.507) IOU(0.018)


## Medium

### Validate with Base

In [15]:
!python val_CITYSCAPES.py --config 'medium' --weight '/kaggle/working/TwinLiteNetPlus/pretrained/medium.pth' \
                --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
                --num_workers 4 \
                --batch_size 16

[CityscapesDataset] 500 images | split='val' | image=(384,640) mask=(360,640)
Total network parameters: 478876
Driving Area Segment: mIOU(0.796)
Lane Line Segment: Acc(0.521) IOU(0.012)


### Finetuning

In [16]:
# !python train_CITYSCAPES.py \
#     --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
#     --hyp "./hyperparameters/twinlitev2_hyper.yaml" \
#     --savedir "./finetune_cityscapes/medium" \
#     --resume "/kaggle/working/TwinLiteNetPlus/pretrained/medium.pth" \
#     --config "medium" \
#     --max_epochs 40 \
#     --batch_size 32 \
#     --num_workers 4 \
#     --ema \
#     --verbose

### Validate with Finetuned

In [17]:
!python val_CITYSCAPES.py --config 'medium' --weight '/kaggle/working/TwinLiteNetPlus/finetune/medium_cityscapes_weights.pth' \
                --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
                --num_workers 4 \
                --batch_size 16

[CityscapesDataset] 500 images | split='val' | image=(384,640) mask=(360,640)
Total network parameters: 478876
/kaggle/working/TwinLiteNetPlus/IOUEval.py:112: RuntimeWarning: invalid value encountered in divide
  IoU = intersection / union
/kaggle/working/TwinLiteNetPlus/IOUEval.py:104: RuntimeWarning: invalid value encountered in divide
  IoU = intersection / union
Driving Area Segment: mIOU(0.923)
Lane Line Segment: Acc(0.512) IOU(0.017)


## Large

### Validate with Base

In [18]:
!python val_CITYSCAPES.py --config 'large' --weight '/kaggle/working/TwinLiteNetPlus/pretrained/large.pth' \
                --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
                --num_workers 4 \
                --batch_size 16

[CityscapesDataset] 500 images | split='val' | image=(384,640) mask=(360,640)
Total network parameters: 1943911
Driving Area Segment: mIOU(0.795)
Lane Line Segment: Acc(0.526) IOU(0.015)


### Finetuning

In [19]:
# !python train_CITYSCAPES.py \
#     --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
#     --hyp "./hyperparameters/twinlitev2_hyper.yaml" \
#     --savedir "./finetune_cityscapes/large" \
#     --resume "/kaggle/working/TwinLiteNetPlus/pretrained/large.pth" \
#     --config "large" \
#     --max_epochs 40 \
#     --batch_size 32 \
#     --num_workers 4 \
#     --ema \
#     --verbose

### Validate with Finetuned

In [20]:
!python val_CITYSCAPES.py --config 'large' --weight '/kaggle/working/TwinLiteNetPlus/finetune/large_cityscapes_weights.pth' \
                --data_root "/kaggle/input/datasets/lqdisme/cityscapes" \
                --num_workers 4 \
                --batch_size 16

[CityscapesDataset] 500 images | split='val' | image=(384,640) mask=(360,640)
Total network parameters: 1943911
/kaggle/working/TwinLiteNetPlus/IOUEval.py:112: RuntimeWarning: invalid value encountered in divide
  IoU = intersection / union
/kaggle/working/TwinLiteNetPlus/IOUEval.py:104: RuntimeWarning: invalid value encountered in divide
  IoU = intersection / union
Driving Area Segment: mIOU(0.934)
Lane Line Segment: Acc(0.531) IOU(0.062)
